# 03 - Exploração da camada Gold
## Tech Challenge Fase 3 — State of Data Brasil (2023, 2024, 2025-2026)

Este notebook lê as 6 tabelas **Parquet** geradas pelo Glue Job `silver_to_gold`
(`s3://.../Gold/mercado/`, `.../remuneracao/`, `.../tecnologias/`, `.../ia/`,
`.../diversidade/`, `.../trabalho/`) e reproduz, em `pandas`, as mesmas análises
de negócio que foram validadas no Athena.

## ⚙️ Rodando no Google Colab

Este notebook lê as 6 tabelas **Gold em Parquet** (pastas `Gold/mercado/`, `Gold/remuneracao/`,
`Gold/tecnologias/`, `Gold/ia/`, `Gold/diversidade/`, `Gold/trabalho/`).

No Colab, a forma mais simples é:

1. Compacte a pasta `Gold/` inteira (com as 6 subpastas) em um `.zip`.
2. Rode a célula abaixo para enviar o zip e descompactar.

In [1]:
# Descomente se estiver no Google Colab:
# !pip install pyarrow -q
# from google.colab import files
# uploaded = files.upload()  # selecione o arquivo Gold.zip
# import zipfile
# with zipfile.ZipFile(list(uploaded.keys())[0]) as z:
#     z.extractall(".")


In [2]:
import pandas as pd

TABELAS_GOLD = ["mercado", "remuneracao", "tecnologias", "ia", "diversidade", "trabalho"]

gold = {}
for nome in TABELAS_GOLD:
    caminho = f"Gold/{nome}"
    gold[nome] = pd.read_parquet(caminho, engine="pyarrow")
    print(f"gold_{nome}: {gold[nome].shape[0]} linhas x {gold[nome].shape[1]} colunas")


gold_mercado: 1885 linhas x 6 colunas
gold_remuneracao: 2429 linhas x 6 colunas
gold_tecnologias: 3303 linhas x 6 colunas
gold_ia: 1074 linhas x 5 colunas
gold_diversidade: 1106 linhas x 6 colunas
gold_trabalho: 495 linhas x 5 colunas


## 1. Schema de cada tabela Gold

In [3]:
for nome, df in gold.items():
    print(f"--- gold_{nome} ---")
    print(list(df.columns))
    print()


--- gold_mercado ---
['regiao', 'senioridade_comparavel', 'cargo_atual', 'nivel_ensino', 'total_profissionais', 'ano_pesquisa']

--- gold_remuneracao ---
['cargo_atual', 'senioridade_comparavel', 'regiao', 'faixa_salarial', 'total_profissionais', 'ano_pesquisa']

--- gold_tecnologias ---
['categoria', 'tecnologia', 'cargo_atual', 'senioridade_comparavel', 'total_mencoes', 'ano_pesquisa']

--- gold_ia ---
['cargo_atual', 'senioridade_comparavel', 'uso_chatgpt_copilot', 'total_profissionais', 'ano_pesquisa']

--- gold_diversidade ---
['genero', 'senioridade_comparavel', 'cargo_atual', 'regiao', 'total_profissionais', 'ano_pesquisa']

--- gold_trabalho ---
['modelo_trabalho', 'faixa_salarial', 'senioridade_comparavel', 'total_profissionais', 'ano_pesquisa']



## 2. MERCADO — total de profissionais e distribuição por senioridade

In [4]:
df = gold["mercado"]

total_por_ano = df.groupby("ano_pesquisa")["total_profissionais"].sum()
print("Total de profissionais por ano:")
print(total_por_ano)


Total de profissionais por ano:
ano_pesquisa
2023         5293
2024         5215
2025-2026    3494
Name: total_profissionais, dtype: int64


In [5]:
senioridade_ano = (
    df.groupby(["ano_pesquisa", "senioridade_comparavel"])["total_profissionais"]
      .sum()
      .unstack()
)
senioridade_pct = senioridade_ano.div(senioridade_ano.sum(axis=1), axis=0).mul(100).round(1)
senioridade_pct


senioridade_comparavel,Júnior,Não informado,Pleno,Sênior
ano_pesquisa,,,,
2023,19.8,27.1,26.3,26.8
2024,16.6,26.8,26.4,30.1
2025-2026,14.8,28.4,22.2,34.5


## 3. REMUNERAÇÃO — faixa salarial mais comum por ano

In [6]:
df = gold["remuneracao"]

faixa_top = (
    df.groupby(["ano_pesquisa", "faixa_salarial"])["total_profissionais"]
      .sum()
      .reset_index()
      .sort_values(["ano_pesquisa", "total_profissionais"], ascending=[True, False])
      .groupby("ano_pesquisa")
      .head(1)
)
faixa_top


,ano_pesquisa,faixa_salarial,total_profissionais
12,2023,de R$ 8.001/mês a R$ 12.000/mês,1026
25,2024,de R$ 8.001/mês a R$ 12.000/mês,1080
38,2025-2026,de R$ 8.001/mês a R$ 12.000/mês,707


## 4. TECNOLOGIAS — top linguagens por ano

In [7]:
df = gold["tecnologias"]

top_linguagens = (
    df[df["categoria"] == "linguagem"]
    .groupby(["ano_pesquisa", "tecnologia"])["total_mencoes"]
    .sum()
    .reset_index()
    .sort_values(["ano_pesquisa", "total_mencoes"], ascending=[True, False])
)
top_linguagens.groupby("ano_pesquisa").head(3)


,ano_pesquisa,tecnologia,total_mencoes
9,2023,Python,3299
10,2023,R,228
13,2023,SQL,74
36,2024,Python,3040
38,2024,R,186
42,2024,SQL,66
79,2025-2026,Python,1928
85,2025-2026,SQL,1763
81,2025-2026,R,294


## 5. INTELIGÊNCIA ARTIFICIAL — evolução da não-adoção

In [8]:
df = gold["ia"]

def eh_nao_usa(valor):
    return "Não utilizo" in str(valor)

df["nao_usa"] = df["uso_chatgpt_copilot"].apply(eh_nao_usa)

# Compativel com qualquer versao de pandas (evita o parametro include_groups,
# disponivel so a partir do pandas 2.2)
nao_usa_por_ano = df[df["nao_usa"]].groupby("ano_pesquisa")["total_profissionais"].sum()
total_por_ano = df.groupby("ano_pesquisa")["total_profissionais"].sum()

resumo_ia = pd.DataFrame({
    "nao_usa": nao_usa_por_ano,
    "total": total_por_ano,
}).fillna(0)
resumo_ia["pct_nao_usa"] = (resumo_ia["nao_usa"] / resumo_ia["total"] * 100).round(1)
resumo_ia

,nao_usa,total,pct_nao_usa
ano_pesquisa,,,
2023,744,3772,19.7
2024,236,3617,6.5
2025-2026,44,2105,2.1


**Validação:** confirma a queda de não-adoção de IA generativa de ~19,7% (2023) para
~2,1% (2025-2026), já usada no storytelling e nas queries SQL.

## 6. DIVERSIDADE — participação de gênero por ano

In [9]:
df = gold["diversidade"]

genero_ano = (
    df.groupby(["ano_pesquisa", "genero"])["total_profissionais"]
      .sum()
      .unstack()
)
genero_pct = genero_ano.div(genero_ano.sum(axis=1), axis=0).mul(100).round(1)
genero_pct


genero,Feminino,Masculino,Outro,Prefiro não informar
ano_pesquisa,,,,
2023,24.4,75.1,0.2,0.3
2024,23.5,76.1,0.2,0.3
2025-2026,22.0,77.5,0.2,0.4


## 7. TRABALHO — evolução do modelo 100% remoto

In [10]:
df = gold["trabalho"]

remoto = (
    df[df["modelo_trabalho"] == "Modelo 100% remoto"]
    .groupby("ano_pesquisa")["total_profissionais"]
    .sum()
)
remoto


ano_pesquisa
2023         2201
2024         2221
2025-2026    1281
Name: total_profissionais, dtype: int64

## 8. Checagem de consistência entre as tabelas Gold

In [11]:
soma_mercado = gold["mercado"]["total_profissionais"].sum()
print(f"Soma de gold_mercado (sem filtro): {soma_mercado}")
print(f"Total esperado (= linhas da Silver, pois mercado não filtra ninguém): 14002")
assert soma_mercado == 14002, "Divergência encontrada!"
print("OK - bate exatamente com o total da Silver.")


Soma de gold_mercado (sem filtro): 14002
Total esperado (= linhas da Silver, pois mercado não filtra ninguém): 14002
OK - bate exatamente com o total da Silver.


## 9. Resumo executivo desta validação

In [12]:
print("Tabelas Gold validadas:")
for nome, df in gold.items():
    print(f"  gold_{nome}: {len(df)} linhas, {df['ano_pesquisa'].nunique()} anos")
print()
print("Conclusão: as 6 tabelas Gold estão íntegras, consistentes entre si, e prontas")
print("para alimentar os gráficos e o material executivo (PPTX).")


Tabelas Gold validadas:
  gold_mercado: 1885 linhas, 3 anos
  gold_remuneracao: 2429 linhas, 3 anos
  gold_tecnologias: 3303 linhas, 3 anos
  gold_ia: 1074 linhas, 3 anos
  gold_diversidade: 1106 linhas, 3 anos
  gold_trabalho: 495 linhas, 3 anos

Conclusão: as 6 tabelas Gold estão íntegras, consistentes entre si, e prontas
para alimentar os gráficos e o material executivo (PPTX).
